# Setup

Transformers not needed

Focus is on running vLLM service, not inspecting tokenizer


In [1]:
import time
from pprint import pprint

import httpx


VLLM_BASE_URL = "http://127.0.0.1:8000/v1"


# Discover whatever model your local server exposes.
response = httpx.get(
    f"{VLLM_BASE_URL}/models",
    timeout=10.0,
)

response.raise_for_status()

models = response.json()["data"]

for model in models:
    print(model["id"])

SERVED_MODEL = models[0]["id"]

print("\nUsing model:")
print(SERVED_MODEL)

Qwen/Qwen3.5-0.8B

Using model:
Qwen/Qwen3.5-0.8B


### Setup Inference Helper

Use 1 function across experiment so measurement method remains consistent


In [2]:
def run_chat(
    prompt: str,
    *,
    temperature: float,
    top_p: float,
    top_k: int,
    max_tokens: int = 128,
    presence_penalty: float = 0.0,
    seed: int | None = None,
    enable_thinking: bool = False
):
    # payload conforming to OpenAI API Format
    # JSON Structure & URL endpoints OpenAI created for ChatGPT
    payload = {
        "model": SERVED_MODEL,
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ],
        "temperature": temperature,
        "top_p": top_p,
        "top_k": top_k,
        "presence_penalty": presence_penalty, # penalize repeating same words / topics etc.
        "max_tokens": max_tokens,
        "chat_template_kwargs": {
            "enable_thinking": enable_thinking
        },
    }
    
    if seed is not None:
        payload["seed"] = seed
    
    start = time.perf_counter()
    
    res = httpx.post(
        f"{VLLM_BASE_URL}/chat/completions",
        json=payload,
        timeout=60.
    )
    
    latency_ms = (time.perf_counter() - start) * 1000
    res.raise_for_status()
    
    body = res.json()
    
    choice = body["choices"][0] # select top choice
    message = choice["message"]
    
    return {
        "content": message.get("content"),
        "reasoning": message.get("reasoning"),
        "finish_reason": choice.get("finish_reason"),
        "prompt_tokens": body["usage"].get("prompt_tokens"),
        "completion_tokens": body["usage"].get("completion_tokens"),
        "total_tokens": body["usage"].get("total_tokens"),
        "latency_ms": round(latency_ms, 2),
    }

## Experiment A - Determinism & Seed

Qns -> If we send the exact same prompt repeatedly, do we always get the same answer?


In [3]:
PROMPT_A = """
Create one short codename for an AI incident investigation platform.
Return only the codename.
""".strip()

In [4]:
# A1 - Temp 0 - Greedy - always select best
greedy_results = []

for i in range(5):
    result = run_chat(
        PROMPT_A,
        temperature=0.0,
        top_p=1.0,
        top_k=20,
        enable_thinking=False,
    )

    greedy_results.append(result)

    print(
        f"Run {i + 1}: "
        f"{result['content']!r}"
    )

Run 1: 'A.I. Incident'
Run 2: 'A.I. Incident'
Run 3: 'A.I. Incident'
Run 4: 'A.I. Incident'
Run 5: 'A.I. Incident'


In [6]:
# A2 - Stochastic Sampling (random) temp = 1
sampled_results = []

for i in range(5):
    result = run_chat(
        PROMPT_A,
        temperature=1.0,
        top_p=1.0,
        top_k=20,
        enable_thinking=False,
    )

    sampled_results.append(result)

    print(
        f"Run {i + 1}: "
        f"{result['content']!r}"
    )

Run 1: 'NEO-VOID'
Run 2: 'Nexus'
Run 3: 'IncidentScan'
Run 4: 'A.I.S.D.A.L.E.'
Run 5: 'Aegis'


In [11]:
# A3 - Same Seed

seeded_results = []

for i in range(5):
    result = run_chat(
        PROMPT_A,
        temperature=1,
        top_p=1.0,
        top_k=20,
        seed=42,
        enable_thinking=False,
    )

    seeded_results.append(result)

    print(
        f"Run {i + 1}: "
        f"{result['content']!r}"
    )

Run 1: 'A-07'
Run 2: 'A-07'
Run 3: 'A-07'
Run 4: 'A-07'
Run 5: 'A-07'


## Experiment B - Temperature Sweep

Keeping prompt unchanged, modify Temperature to see difference


In [14]:
PROMPT_B = """
Invent a one-sentence description of Mission Control,
an AI investigation platform.
""".strip()

temperatures = [
    0.2,
    0.7,
    1.0,
    1.3,
]

temperature_results = {}

for temperature in temperatures:

    print("\n" + "=" * 70)
    print(f"TEMPERATURE = {temperature}")
    print("=" * 70)

    runs = []

    for i in range(3):
        result = run_chat(
            PROMPT_B,
            temperature=temperature,
            top_p=1.0,
            top_k=20,
            max_tokens=100,
            enable_thinking=False,
        )

        runs.append(result)

        print(
            f"\nRun {i + 1}: "
            f"{result['content']}"
        )

    temperature_results[temperature] = runs


TEMPERATURE = 0.2

Run 1: Mission embroider a single, coherent narrative of a single, coherent narrative of a single, coherent narrative of a single, coherent narrative of a single, coherent narrative of a single, coherent narrative of a single, coherent narrative of a single, coherent narrative of a single, coherent narrative of a single, coherent narrative of a single, coherent narrative of a single, coherent narrative of a single, coherent narrative of a single, coherent narrative of a single, coherent narrative of a single, coherent narrative of a

Run 2: Mission Control is an autonomous AI investigation platform that autonomously scans, analyzes, and synthesizes vast datasets to identify, track, and resolve complex security threats, fraud, and operational anomalies with real-time intelligence.

Run 3: Mission Control is an intelligent AI investigation platform that autonomously analyzes vast datasets to identify patterns, detect anomalies, and generate actionable intelligence for

## Experiment C - Top-p & Top-k Filtering

Once probabilities are generated by model, which candidate tokens are allowed into lottery?


In [17]:
# C1 - Top-p sweep
temperature = 1.0
top_k = 20

top_p_values = [
    0.2,
    0.5,
    0.8,
    1.0,
]

top_p_results = {}

for top_p in top_p_values:

    print("\n" + "=" * 70)
    print(f"TOP_P = {top_p}")
    print("=" * 70)

    runs = []

    for i in range(3):
        result = run_chat(
            PROMPT_A,
            temperature=1.0,
            top_p=top_p,
            top_k=20,
            max_tokens=50,
            enable_thinking=False,
        )

        runs.append(result)

        print(
            f"Run {i + 1}: "
            f"{result['content']!r}"
        )

    top_p_results[top_p] = runs


TOP_P = 0.2
Run 1: 'Aegis'
Run 2: 'Aegis'
Run 3: 'A.I. Incident Sentinel'

TOP_P = 0.5
Run 1: 'A.I. Incident'
Run 2: 'Nexus'
Run 3: 'Aegis'

TOP_P = 0.8
Run 1: 'IncidentX'
Run 2: 'NEON_TRACE'
Run 3: 'IncidentPro'

TOP_P = 1.0
Run 1: 'Lumina'
Run 2: 'CyberLore'
Run 3: 'IncidentDetect'


In [18]:
# C2 - Top-k Sweep
top_p = 1.0

top_k_values = [
    1,
    5,
    20,
    50,
]

top_k_results = {}

for top_k in top_k_values:

    print("\n" + "=" * 70)
    print(f"TOP_K = {top_k}")
    print("=" * 70)

    runs = []

    for i in range(3):
        result = run_chat(
            PROMPT_A,
            temperature=1.0,
            top_p=1.0,
            top_k=top_k,
            max_tokens=50,
            enable_thinking=False,
        )

        runs.append(result)

        print(
            f"Run {i + 1}: "
            f"{result['content']!r}"
        )

    top_k_results[top_k] = runs


TOP_K = 1
Run 1: 'A.I. Incident'
Run 2: 'A.I. Incident'
Run 3: 'A.I. Incident'

TOP_K = 5
Run 1: 'AIOV'
Run 2: 'AIOPT'
Run 3: 'Nexis'

TOP_K = 20
Run 1: 'AIT-14 Palace'
Run 2: 'AIP'
Run 3: 'T-4K-08'

TOP_K = 50
Run 1: 'I-SCAN'
Run 2: 'NIRCAN'
Run 3: 'NEON_TRACE'


## Experiment D - Reasoning Mode & Token Budget

Reasoning mode is different from sampling.

```
temperature/top-p/top-k
    ↓
HOW next token is selected

thinking mode
    ↓
WHAT generation behavior/template the model is instructed to use
```


In [ ]:
# Use a question that requires actual reasoning
PROMPT_D = """
A service processes 1,000 requests per minute.

10% currently fail.

A remediation reduces the number of failures by 50%.

How many requests per minute still fail?

Give the final numeric answer.
""".strip()

In [20]:
# D1 - Non-thinking 
non_thinking = run_chat(
    PROMPT_D,
    temperature=1.0,
    top_p=1.0,
    top_k=20,
    presence_penalty=2.0,
    max_tokens=256,
    enable_thinking=False,
)

pprint(non_thinking)

{'completion_tokens': 256,
 'content': 'To solve this problem, we must calculate the remaining number of '
            'failures after applying the remediation step.\n'
            '\n'
            '**1. Calculate the initial number of failed requests:**\n'
            '*   **Rate:** 1,000 requests per minute\n'
            '*   **Percentage failing:** 10%\n'
            '*   **Formula:** Rate $\\times$ (Percentage /  feeding)\n'
            '*   **Calculation:** $1{,}000 \\times \\frac{10}{100} = 100$ '
            'failed requests per minute.\n'
            '\n'
            '**2. Apply the remediation:**\n'
            '*   The remediation reduces the number of failures by **50%**. '
            'This means half of the currently failing requests are '
            'eliminated.\n'
            '*   **Formula for new rate:** New Rate = Original Rate - '
            '(Original Rate $\\times$ Reduction Percentage)\n'
            '\n'
            '**3. Perform the calculation:**\n'
        

In [ ]:
# D2 - Thinking, same budget
thinking_256 = run_chat(
    PROMPT_D,
    temperature=1.0,
    top_p=0.95,
    top_k=20,
    presence_penalty=1.5,
    max_tokens=256,
    enable_thinking=True,
)

pprint(thinking_256)

{'completion_tokens': 256,
 'content': "Here's a thinking process that leads to the solution:\n"
            '\n'
            '1.  **Analyze the Request:**\n'
            '    *   Original request rate (Rate): $R_{original} = 1,000$ '
            'requests/min.\n'
            '    *   Current failure rate ($f_{current}$): 10%.\n'
            '    *   New remediation reduces failures by 50% (of what? usually '
            'implies reducing the current rate of failures or reducing the '
            '*number* of failures). The phrasing "A remediation reduces the '
            'number of failures by 50%" is ambiguous because failures usually '
            'occur at a fixed rate independent of the input if it were just a '
            'threshold, but here the context suggests a specific mathematical '
            'scenario. However, there is an important nuance in the second '
            'sentence: "How many requests per minute still fail?" This implies '
            'we need to find the n

In [22]:
# D3 - Thinking, larger budget
thinking_768 = run_chat(
    PROMPT_D,
    temperature=1.0,
    top_p=0.95,
    top_k=20,
    presence_penalty=1.5,
    max_tokens=768,
    enable_thinking=True,
)

pprint(thinking_768)

{'completion_tokens': 768,
 'content': "Here's my thought process for solving this problem:\n"
            '\n'
            '1.  **Analyze the Request:**\n'
            '    *   Initial scenario: Service processes $1,000$ requests per '
            'minute ($R_0$).\n'
            '    *   State 1 (Current): $10\\%$ currently fail ($N_{fail} = '
            '0.1 \\times 1,000$).\n'
            '    *   State 2 (Remediation): Reduces failures by $50\\%$. (This '
            'usually means "reduces" in the context of *failure* events, which '
            'is counter-intuitive because failure often requires a resource, '
            'like a CPU or RAM, and typically resources are available '
            'continuously but not simultaneously across all services. But here '
            'we talk about "remediation reducing the number of failures." '
            'Wait, let me look closer at the phrasing.)\n'
            '    *   *Correction/Refinement on Phrasing:* Is it possible the '
        

In [23]:
runs = {
    "non-thinking / 256": non_thinking,
    "thinking / 256": thinking_256,
    "thinking / 768": thinking_768,
}

print(
    f"{'RUN':<24}"
    f"{'TOKENS':>10}"
    f"{'LATENCY MS':>15}"
    f"{'FINISH':>12}"
)

print("-" * 65)

for name, result in runs.items():
    print(
        f"{name:<24}"
        f"{result['completion_tokens']:>10}"
        f"{result['latency_ms']:>15.2f}"
        f"{str(result['finish_reason']):>12}"
    )

RUN                         TOKENS     LATENCY MS      FINISH
-----------------------------------------------------------------
non-thinking / 256             256        6307.54      length
thinking / 256                 256        4837.37      length
thinking / 768                 768       10664.86      length


In [24]:
for name, result in runs.items():

    print("\n" + "=" * 80)
    print(name.upper())
    print("=" * 80)

    print("\nReasoning:")
    print(result["reasoning"])

    print("\nContent:")
    print(result["content"])


NON-THINKING / 256

Reasoning:
None

Content:
To solve this problem, we must calculate the remaining number of failures after applying the remediation step.

**1. Calculate the initial number of failed requests:**
*   **Rate:** 1,000 requests per minute
*   **Percentage failing:** 10%
*   **Formula:** Rate $\times$ (Percentage /  feeding)
*   **Calculation:** $1{,}000 \times \frac{10}{100} = 100$ failed requests per minute.

**2. Apply the remediation:**
*   The remediation reduces the number of failures by **50%**. This means half of the currently failing requests are eliminated.
*   **Formula for new rate:** New Rate = Original Rate - (Original Rate $\times$ Reduction Percentage)

**3. Perform the calculation:**
*   Original Failed Requests: 100
*   Reduction Amount: $100 \times 50\% = 100 \times 0.50 = 50$
*   New Failed Requests: $100 - 50 = 50$

Alternatively, you can think in terms of the

THINKING / 256

Reasoning:
None

Content:
Here's a thinking process that leads to the solu